# Muestreo estratificado de Amazon Reviews'23 (streaming, sin bajar el raw completo)

Este notebook arma una muestra estratificada local en Parquet a partir del dataset completo (377GB entre reviews y metadata en HuggingFace), **sin** descargar los archivos crudos completos: usa `streaming=True` para leer fila por fila desde `hf://` vía HTTP y aplica *reservoir sampling* (Algoritmo R de Vitter) por estrato `(categoría, rating)`, de forma que cada fila tiene la misma probabilidad de terminar en la muestra sin necesitar conocer de antemano el tamaño del archivo.

Después de samplear las reviews, se levanta la metadata de producto (`raw/meta_categories`) filtrando solo los `parent_asin` que aparecen en la muestra — así reviews y metadata quedan consistentes (se puede hacer join) sin tener que samplear ni bajar toda la metadata.

**Costo/tiempo**: se lee una vez el volumen completo de las categorías elegidas (no hay forma de evitarlo para que el muestreo sea representativo), pero no se persiste nada en disco salvo la muestra final. Recomendado correrlo desde una máquina con buen ancho de banda (ver nota del server Ubuntu en el README/plan del proyecto).

**Nota**: para esta corrida de prueba se usan 3 categorías chicas (`All_Beauty`, `Amazon_Fashion`, `Software`, ~3GB en total) para validar el pipeline rápido. Para el dataset real del proyecto, cambiar `CATEGORIES` por el subconjunto de categorías relevantes (ver celda de configuración).

In [1]:
import json
import random
from pathlib import Path

import pandas as pd
import requests
from huggingface_hub import hf_hub_url

## Configuración

In [2]:
# Categorías a muestrear. Para la corrida real del proyecto, reemplazar por el subconjunto
# relevante de retail, por ejemplo:
# CATEGORIES = [
#     "Electronics", "Beauty_and_Personal_Care", "Health_and_Household",
#     "Grocery_and_Gourmet_Food", "Toys_and_Games", "Office_Products", "Pet_Supplies",
# ]
CATEGORIES = ["All_Beauty", "Amazon_Fashion", "Software"]

# Tamaño del reservorio por estrato (categoria, rating 1-5). Con 5 ratings x N categorias,
# el tope teórico de filas en la muestra de reviews es 5 * N * RESERVOIR_SIZE_PER_STRATUM.
RESERVOIR_SIZE_PER_STRATUM = 2000

RANDOM_SEED = 42

OUTPUT_DIR = Path("../data/samples")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_ID = "McAuley-Lab/Amazon-Reviews-2023"
REVIEWS_PATH_TMPL = "raw/review_categories/{category}.jsonl"
META_PATH_TMPL = "raw/meta_categories/meta_{category}.jsonl"


def stream_jsonl(repo_path: str):
    """Itera línea a línea un .jsonl del repo vía HTTP streaming, sin persistir nada en disco.

    Se evita a propósito `datasets.load_dataset(..., streaming=True)` para este caso: al inferir
    el esquema Arrow de a chunks, algunas categorías (p. ej. metadata de `Software`, que agrega
    campos como `author`/`subtitle` ausentes en otras categorías) rompen el cast con
    `CastError: ... because column names don't match`. Leer con `requests` + `json.loads` fila a
    fila evita depender de un esquema homogéneo entre categorías.
    """
    url = hf_hub_url(REPO_ID, repo_path, repo_type="dataset")
    with requests.get(url, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        for line in resp.iter_lines():
            if line:
                yield json.loads(line)

## Reservoir sampling estratificado por rating (1 pasada, streaming)

In [3]:
def stratified_reservoir_sample_reviews(category: str, reservoir_size: int, seed: int) -> pd.DataFrame:
    """Reservoir sampling (Algoritmo R) independiente por cada valor de rating (1-5)."""
    rng = random.Random(seed)
    reservoirs = {r: [] for r in range(1, 6)}
    seen = {r: 0 for r in range(1, 6)}

    for row in stream_jsonl(REVIEWS_PATH_TMPL.format(category=category)):
        rating = int(row["rating"])
        if rating not in reservoirs:
            continue
        seen[rating] += 1
        reservoir = reservoirs[rating]
        if len(reservoir) < reservoir_size:
            reservoir.append(row)
        else:
            j = rng.randint(0, seen[rating] - 1)
            if j < reservoir_size:
                reservoir[j] = row

    rows = [row for reservoir in reservoirs.values() for row in reservoir]
    df = pd.DataFrame(rows)
    df["category"] = category
    print(f"{category}: vistas {sum(seen.values()):,} reviews, muestreadas {len(df):,}")
    return df

In [4]:
reviews_samples = [
    stratified_reservoir_sample_reviews(category, RESERVOIR_SIZE_PER_STRATUM, RANDOM_SEED)
    for category in CATEGORIES
]
reviews_df = pd.concat(reviews_samples, ignore_index=True)
reviews_df.head()

All_Beauty: vistas 701,528 reviews, muestreadas 10,000


Amazon_Fashion: vistas 2,500,939 reviews, muestreadas 10,000


Software: vistas 4,880,181 reviews, muestreadas 10,000


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,category
0,1.0,Terrible lashes,They are way different then the picture they a...,[],B07MSDRY4K,B07MSDRY4K,AFQFCIAPBWCC76STIBEXVEQGGWAQ,1556571815191,6,True,All_Beauty
1,1.0,Wouldn’t dry or go on even.,This was awful,[],B07N37TMK7,B07N37TMK7,AE7GWJAPV4ULU6XVJJAYDBRJTFGQ,1562794952019,0,True,All_Beauty
2,1.0,Doesn’t work!,It worked for about a week and then would no l...,[],B07XY659KF,B0C17MLTNF,AF4Q2VXNN6CED7G2EZG2GVLPV2WQ,1614474408482,0,True,All_Beauty
3,1.0,It is the wrong color,I ordered a color that was supposed to be yell...,[],B07MSD2NRS,B07BW9CGXX,AEUINLTGB3RRD2JRICBMHSYTIVEQ,1571703153731,0,True,All_Beauty
4,1.0,These nails are not sturdy at all,"In fairness, I bought mine at Walgreens, and a...",[],B01N3JNUJ5,B01N3JNUJ5,AHCWJNV4HY5UIAGV6BYNWRCWD54Q,1556332821846,4,False,All_Beauty


## Metadata de los productos presentes en la muestra (join-consistent)

In [5]:
def metadata_for_sampled_asins(category: str, asins: set) -> pd.DataFrame:
    """Filtra, en una sola pasada streaming, solo los parent_asin presentes en la muestra de reviews."""
    rows = [
        row
        for row in stream_jsonl(META_PATH_TMPL.format(category=category))
        if row["parent_asin"] in asins
    ]
    df = pd.DataFrame(rows)
    df["category"] = category
    print(f"{category}: {len(asins):,} asins buscados, {len(df):,} encontrados en metadata")
    return df

In [6]:
meta_samples = []
for category in CATEGORIES:
    asins = set(reviews_df.loc[reviews_df["category"] == category, "parent_asin"])
    meta_samples.append(metadata_for_sampled_asins(category, asins))
meta_df = pd.concat(meta_samples, ignore_index=True)
meta_df.head()

All_Beauty: 7,476 asins buscados, 7,476 encontrados en metadata


Amazon_Fashion: 9,300 asins buscados, 9,300 encontrados en metadata


Software: 4,183 asins buscados, 4,183 encontrados en metadata


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,category
0,All Beauty,10 PCS Professional Hair Cutting Scissors 6.7i...,4.3,128.0,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Astraet,[],"{'Color': 'Pink', 'Material': 'Rubber', 'Brand...",B088ZB4DTH,None,All_Beauty
1,All Beauty,7 Packs Deep Wave Crochet Hair 22 Inch Deep wa...,3.4,10.0,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Yun Mei Hair,[],"{'Brand': 'Yun Mei Hair', 'Material': 'Synthet...",B07Z818MLY,None,All_Beauty
2,All Beauty,"Zydeco Chop Chop Cajun Seasoning Base, 8 Ounce...",4.7,21.0,"[All Natural blend of Dehydrated Onion, Dehydr...",[Zydeco Chop Chop is a blend of Dehydrated Oni...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],BORELTH,[],{'Package Dimensions': '9.61 x 7.17 x 3.07 inc...,B0BTLTVR1X,None,All_Beauty
3,All Beauty,Sharonelle Natural Cream Soft Wax for Sensitiv...,4.0,1.0,[],[],50.0,[{'thumb': 'https://m.media-amazon.com/images/...,[],Sharonelle,[],"{'Item Form': 'Wax', 'Skin Type': 'Sensitive',...",B071G4NNH7,None,All_Beauty
4,All Beauty,Beauty Wig World 4Pcs Floral Print Scarf Scrun...,4.3,287.0,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],pureluca,[],"{'Material': 'Silk', 'Item Form': 'Scrunchie',...",B07QKYM8LR,None,All_Beauty


## Export a Parquet local

In [7]:
reviews_path = OUTPUT_DIR / "reviews_sample.parquet"
meta_path = OUTPUT_DIR / "meta_sample.parquet"

reviews_df.to_parquet(reviews_path, index=False)
meta_df.to_parquet(meta_path, index=False)

print(f"reviews: {len(reviews_df):,} filas -> {reviews_path} ({reviews_path.stat().st_size / 1e6:.1f} MB)")
print(f"metadata: {len(meta_df):,} filas -> {meta_path} ({meta_path.stat().st_size / 1e6:.1f} MB)")

reviews: 30,000 filas -> ..\data\samples\reviews_sample.parquet (5.5 MB)
metadata: 20,959 filas -> ..\data\samples\meta_sample.parquet (14.1 MB)


In [8]:
reviews_df.groupby(["category", "rating"]).size().unstack(fill_value=0)

rating,1.0,2.0,3.0,4.0,5.0
category,,,,,
All_Beauty,2000,2000,2000,2000,2000
Amazon_Fashion,2000,2000,2000,2000,2000
Software,2000,2000,2000,2000,2000
